<a href="https://colab.research.google.com/github/Jay-D21/Collage-Work/blob/ATML/ATML_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. Training and Network Parameters
learning_rate = 0.0002
batch_size = 128
epochs = 100000
image_dim = 784  # 28x28 pixels
gen_hidd_dim = 256
disc_hidd_dim = 256
z_noise_dim = 100

In [ ]:
# 2. Data Preparation
(x_train, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.reshape(-1, image_dim).astype('float32') / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(batch_size)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# 3. Define the Network Architecture
# Use  of Keras Sequential models to replicate the layers in the source
def build_generator():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(gen_hidd_dim, activation='relu', input_shape=(z_noise_dim,)),
        tf.keras.layers.Dense(image_dim, activation='sigmoid')
    ])
    return model

# --- ADDED ESSENTIAL CHANGES HERE ---
def build_discriminator():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(disc_hidd_dim, activation='relu', input_shape=(image_dim,)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

generator = build_generator()
discriminator = build_discriminator()
# ------------------------------------


In [ ]:
# 4. Optimizers
gen_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

In [ ]:
# 5. Training Step with Adversarial Loss

# --- ADDED ESSENTIAL CHANGES HERE ---
@tf.function
# ------------------------------------
def train_step(real_images):
    # Generate random noise to feed the generator
    noise = tf.random.uniform([batch_size, z_noise_dim], -1., 1.)

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate fake images
        fake_images = generator(noise, training=True)

        # Discriminator outputs
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)

        # Loss Logic from Source 2:
        # Disc_Loss = -mean(log(D(x)) + log(1 - D(G(z))))
        # Gen_Loss = -mean(log(D(G(z))))
        disc_loss = -tf.reduce_mean(tf.math.log(real_output + 1e-10) + tf.math.log(1. - fake_output + 1e-10))
        gen_loss = -tf.reduce_mean(tf.math.log(fake_output + 1e-10))

    # --- ADDED ESSENTIAL CHANGES HERE ---
    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_gradients, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_gradients, discriminator.trainable_variables))
    
    return gen_loss, disc_loss
    # ------------------------------------


In [ ]:
# 6. Training Loop
# --- ADDED ESSENTIAL CHANGES HERE ---
dataset_iter = iter(train_dataset)
# ------------------------------------

for epoch in range(epochs):
    # --- ADDED ESSENTIAL CHANGES HERE ---
    # Get a batch of real images
    try:
        real_batch = next(dataset_iter)
    except StopIteration:
        dataset_iter = iter(train_dataset)
        real_batch = next(dataset_iter)
    # ------------------------------------

    # --- ADDED ESSENTIAL CHANGES HERE ---
    # Check if batch size is correct (for the last partial batch)
    if real_batch.shape[0] != batch_size:
        continue
    # ------------------------------------

    g_loss, d_loss = train_step(real_batch)

    if epoch % 2000 == 0:
        print(f"Steps: {epoch} : Generator Loss: {g_loss:.4f}, Discriminator Loss: {d_loss:.4f}")


In [ ]:
# 7. Testing & Visualization
n = 6
canvas = np.empty((28 * n, 28 * n))
# --- ADDED ESSENTIAL CHANGES HERE ---
noise = tf.random.uniform([n * n, z_noise_dim], -1., 1.)
# ------------------------------------
generated_images = generator(noise, training=False).numpy()

for i in range(n):

    for j in range(n):
        # --- ADDED ESSENTIAL CHANGES HERE ---
        img = generated_images[i * n + j].reshape(28, 28)
        # ------------------------------------
        canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = img

plt.figure(figsize=(n, n))
plt.imshow(canvas, origin="upper", cmap="gray")
plt.show()
